# POC for distributed data alignment


## Preparation of the dataset

We use Adult as typical benchmark dataset.

### Dataset loading

We are only interested in the features

In [1]:
import os
import sys
from pathlib import Path

base_path = Path(os.path.abspath("")).parent.joinpath("src")
sys.path.append(base_path.as_posix())

In [2]:
# !uv pip install kagglehub

In [3]:
from pathlib import Path

import kagglehub
import pandas as pd

dataset_names = {
    "georgesaavedra/covid19-dataset": "owid-covid-data.csv",
    "hendratno/covid19-indonesia": "covid_19_indonesia_time_series_all.csv",
}
datasets = {}
for dataset_name, file_name in dataset_names.items():
    path = kagglehub.dataset_download(dataset_name)
    csv_path = Path(path).joinpath(file_name).as_posix()
    print(csv_path)
    datasets[dataset_name] = pd.read_csv(csv_path)

/Users/stefano/Projects/PrivFusion/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/Users/stefano/.cache/kagglehub/datasets/georgesaavedra/covid19-dataset/versions/6/owid-covid-data.csv
/Users/stefano/.cache/kagglehub/datasets/hendratno/covid19-indonesia/versions/86/covid_19_indonesia_time_series_all.csv


### Parties creation

Let's split the dataset and create different parties relying on the same schema (to begin with)

In [4]:
A, B = datasets["georgesaavedra/covid19-dataset"], datasets["hendratno/covid19-indonesia"]

And let us augment the dataset adding a date column, for no particular reason other than it is useful

In [5]:
import datetime
import random


def generate_dates(many: int, format: str = r"%Y-%m-%d") -> list[str]:
    return [
        datetime.date(
            int(1980 + 50 * random.random()), int(6 + random.random() * 6), int(15 + 10 * random.random())
        ).strftime(format)
        for _ in range(many)
    ]

In [6]:
A.head()

,iso_code,continent,location,date,total_cases,new_cases,new_cases_smoothed,total_deaths,new_deaths,new_deaths_smoothed,...,female_smokers,male_smokers,handwashing_facilities,hospital_beds_per_thousand,life_expectancy,human_development_index,excess_mortality_cumulative_absolute,excess_mortality_cumulative,excess_mortality,excess_mortality_cumulative_per_million
0,AFG,Asia,Afghanistan,2020-02-24,5.0,5.0,NaN,NaN,NaN,NaN,...,NaN,NaN,37.746,0.5,64.83,0.511,NaN,NaN,NaN,NaN
1,AFG,Asia,Afghanistan,2020-02-25,5.0,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,37.746,0.5,64.83,0.511,NaN,NaN,NaN,NaN
2,AFG,Asia,Afghanistan,2020-02-26,5.0,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,37.746,0.5,64.83,0.511,NaN,NaN,NaN,NaN
3,AFG,Asia,Afghanistan,2020-02-27,5.0,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,37.746,0.5,64.83,0.511,NaN,NaN,NaN,NaN
4,AFG,Asia,Afghanistan,2020-02-28,5.0,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,37.746,0.5,64.83,0.511,NaN,NaN,NaN,NaN


In [7]:
B.head()

,Date,Location ISO Code,Location,New Cases,New Deaths,New Recovered,New Active Cases,Total Cases,Total Deaths,Total Recovered,...,Latitude,New Cases per Million,Total Cases per Million,New Deaths per Million,Total Deaths per Million,Total Deaths per 100rb,Case Fatality Rate,Case Recovered Rate,Growth Factor of New Cases,Growth Factor of New Deaths
0,3/1/2020,ID-JK,DKI Jakarta,2,0,0,2,39,20,75,...,-6.204699,0.18,3.60,0.0,1.84,0.18,51.28%,192.31%,NaN,NaN
1,3/2/2020,ID-JK,DKI Jakarta,2,0,0,2,41,20,75,...,-6.204699,0.18,3.78,0.0,1.84,0.18,48.78%,182.93%,1.0,1.0
2,3/2/2020,IDN,Indonesia,2,0,0,2,2,0,0,...,-0.789275,0.01,0.01,0.0,0.00,0.00,0.00%,0.00%,NaN,NaN
3,3/2/2020,ID-RI,Riau,1,0,0,1,1,0,1,...,0.511648,0.16,0.16,0.0,0.00,0.00,0.00%,100.00%,NaN,NaN
4,3/3/2020,ID-JK,DKI Jakarta,2,0,0,2,43,20,75,...,-6.204699,0.18,3.96,0.0,1.84,0.18,46.51%,174.42%,1.0,1.0


In [8]:
from privfusion.agents.llms import OllamaLLM
from privfusion.dataset_analyzer import DatasetAnalyzer

llm = OllamaLLM("gpt-oss:20b")

analyzer = DatasetAnalyzer(llm, {})

In [ ]:
sem_info_a = analyzer.extract_information_from_dataset("A", A)
sem_info_a

In [ ]:
sem_info_b = analyzer.extract_information_from_dataset("B", B)
sem_info_b